# Week 3 · Part 2 — Compare models fairly with cross-validation

**Question for today:** which model predicts price best, and how sure can we be, **without touching the test rows**?

In Part 1 each model got **one** score from 83 test rows. Today we check how much such a score can change, and we learn a fairer way to compare models.

### Plan
1. Setup: the same data and the same split as Part 1
2. Is one test score reliable?
3. Let scikit-learn run the folds: `cross_validate`, RMSE, and R²
4. A fair comparison of four models
5. Compare fold by fold
6. What do the coefficients say?
7. Where does the model miss? Out-of-fold residuals
8. Your turn

### Words you will meet
| Word | Meaning |
|---|---|
| fold | one part of the training rows, used once for validation |
| cross-validation | fit on some folds, score on the remaining fold, repeat for every fold |
| RMSE | root mean squared error: like MAE, but large misses count more |
| R² | how much better than predicting the mean, on a scale where 1 is perfect |
| out-of-fold prediction | a prediction for a row made by a model that did not see that row |

## 1. Setup: same data, same split
We load the tools. New ones today: `KFold`, `cross_validate`, `cross_val_predict`, `root_mean_squared_error`, and `r2_score`; each is explained when we first use it. `Pipeline`, `ColumnTransformer`, and `StandardScaler` are familiar from Week 2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer          # choose or transform input columns
from sklearn.dummy import DummyRegressor               # baseline: always predict the training mean
from sklearn.linear_model import LinearRegression      # least-squares regression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline                  # chain preprocessing and model
from sklearn.preprocessing import FunctionTransformer, StandardScaler

Load the data. `features` lists the six input columns; `record_id` is left out because it is only a row label.

In [ ]:
df = pd.read_csv("Dataset.csv")
target = "price_10k_twd_per_ping"
features = ["transaction_date_decimal", "house_age_years", "distance_to_mrt_m",
            "convenience_stores", "latitude", "longitude"]

Now **exactly the same split** as Part 1 (same `test_size`, same `random_state`), so the 83 test rows are the same rows.

From now on the test rows are **locked**: every comparison in this notebook uses training rows only. The cell also creates `cv`, which divides the training rows into five folds; section 2 explains it.

In [ ]:
train, test = train_test_split(df, test_size=0.20, random_state=42)
X_train, y_train = train[features], train[target]
cv = KFold(n_splits=5, shuffle=True, random_state=42)
print("training rows:", len(train), "  locked test rows:", len(test))

## 2. Is one test score reliable?
The test set has 83 rows. With a different random 83 rows, the same model would get a different MAE. How different?

We can find out using the training rows only:
1. Divide the 331 training rows into 5 parts, called **folds**.
2. For each fold: fit the model on the other 4 folds and score it on this fold.
3. Every training row is used for scoring exactly once.

This is **5-fold cross-validation**, as in Week 2.

**New tool — `KFold(n_splits=5, shuffle=True, random_state=42)`.** In Week 2 you used `StratifiedKFold`, which keeps class proportions equal in every fold. A regression target has no classes, so we use plain `KFold`. `shuffle=True` mixes the rows first; `random_state=42` makes the folds the same every time.

First, just the fold sizes:

In [ ]:
for fold, (fit_rows, check_rows) in enumerate(cv.split(X_train), start=1):
    print(f"fold {fold}: fit on {len(fit_rows)} rows, score on {len(check_rows)} rows")

**What we see:** each fold scores about 66 rows, and the model is fitted on the other 265.

Now the distance-only line from Part 1, fitted and scored once per fold. Before running, **predict**: will the five MAE values be almost equal or quite different?

In [ ]:
X_distance = X_train[["distance_to_mrt_m"]]
fold_mae = []
for fit_rows, check_rows in cv.split(X_distance):
    line = LinearRegression().fit(X_distance.iloc[fit_rows], y_train.iloc[fit_rows])
    predicted = line.predict(X_distance.iloc[check_rows])
    fold_mae.append(mean_absolute_error(y_train.iloc[check_rows], predicted))
print("MAE on each fold:", np.round(fold_mae, 2))
print("mean:", round(np.mean(fold_mae), 3), "  SD:", round(np.std(fold_mae, ddof=1), 3))

**What we see:** the **same model** scores between 6.29 and 8.96 depending on which rows are scored. The mean is 7.48 and the fold-to-fold SD is about 1.05.

So a single score is **one draw**. If two models differ by 0.3 on one split, that difference may be luck. From now on we report the **mean over folds** and look at the **spread**.

## 3. Let scikit-learn run the folds
**New tool — `cross_validate(model, X, y, cv=cv, scoring=...)`.** It runs the loop above for us: it fits a fresh copy of the model on each fold and returns the scores.

We ask for three scores. MAE you know. Two are new:

- **RMSE** (root mean squared error): square each error, average, take the square root.
  $$\text{RMSE} = \sqrt{\tfrac{1}{n}\sum (y_i - \hat{y}_i)^2}$$
  It is also in price units, but squaring makes **large misses count more**. So RMSE is never smaller than MAE, and a big gap between them means some misses are very large.
- **R²** compares the model's squared error with the squared error of simply predicting the mean of the scored rows:
  $$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$
  R² = 1 is perfect, R² = 0 is no better than the mean, and R² < 0 is worse than the mean. It is **not** a percentage of correct answers.

The `scoring` names tell scikit-learn which scores to compute:

In [ ]:
scoring = {"MAE": "neg_mean_absolute_error",
           "RMSE": "neg_root_mean_squared_error",
           "R2": "r2"}

In [ ]:
result = cross_validate(LinearRegression(), X_distance, y_train, cv=cv, scoring=scoring)
pd.DataFrame(result)[["test_MAE", "test_RMSE", "test_R2"]].round(3)

**What we see:** the MAE and RMSE columns are **negative**. scikit-learn always treats a larger score as better, so it stores errors with a minus sign (`neg_mean_absolute_error`). We simply flip the sign back.

The five MAE values are the same as in our own loop, because `cv` gives the same folds.

In [ ]:
print("CV MAE: ", round(-result["test_MAE"].mean(), 3))
print("CV RMSE:", round(-result["test_RMSE"].mean(), 3))
print("CV R²:  ", round(result["test_R2"].mean(), 3))

**What we see:** MAE 7.476, the same as our loop. RMSE 10.21 is much larger than MAE, so some misses are big. R² 0.43: the line removes a good part of the squared error that the mean would leave, but far from all of it.

## 4. A fair comparison of four models
A comparison is fair when every model gets **the same rows, the same folds, and the same metrics**. We compare four candidates:

| Candidate | Inputs |
|---|---|
| mean dummy | none |
| distance only | `distance_to_mrt_m` |
| stores only | `convenience_stores` |
| all six inputs | all six input columns |

To keep the protocol identical, every model receives the **same six-column table** `X_train`, and each model picks its own columns. The small helper `uses(...)` does that with a `ColumnTransformer` (Week 2) that simply passes the named columns through. Wrapped in a `Pipeline`, column choice and model are fitted together inside every fold.

In [ ]:
def uses(columns):
    """Pass only the named input columns to the model."""
    return ColumnTransformer([("keep", "passthrough", columns)])

In [ ]:
models = {
    "mean dummy": DummyRegressor(strategy="mean"),
    "distance only": Pipeline([("inputs", uses(["distance_to_mrt_m"])), ("model", LinearRegression())]),
    "stores only": Pipeline([("inputs", uses(["convenience_stores"])), ("model", LinearRegression())]),
    "all six inputs": Pipeline([("inputs", uses(features)), ("model", LinearRegression())]),
}

The next helper, `evaluate(models)`, is section 3 in a loop: for each model it runs `cross_validate`, flips the error signs, and collects
- a **summary table**: mean MAE, fold-to-fold SD of MAE, mean RMSE, mean R²;
- a **fold table**: the MAE of every model on every fold.

In [ ]:
def evaluate(models):
    """Cross-validate every model on the same five training folds."""
    rows, fold_mae = [], {}
    for name, model in models.items():
        result = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
        mae = -result["test_MAE"]
        fold_mae[name] = mae
        rows.append({"model": name,
                     "MAE": mae.mean(),
                     "MAE fold SD": mae.std(ddof=1),
                     "RMSE": -result["test_RMSE"].mean(),
                     "R2": result["test_R2"].mean()})
    table = pd.DataFrame(rows).set_index("model").round(3)
    folds = pd.DataFrame(fold_mae, index=[f"fold {i}" for i in range(1, 6)]).T.round(2)
    return table, folds

Before running: **rank the four models by MAE**, best first.

In [ ]:
table, folds = evaluate(models)
table

**What we see:**
- The dummy misses by 10.70. Every real model beats it.
- One input: distance (7.48) is better than store count (8.53).
- All six inputs together are best: MAE 6.55, R² 0.55.
- The dummy's R² is slightly **below zero** (−0.026). It predicts the mean of the rows it was fitted on, while R² on each fold compares with that fold's own mean. It is a little worse than that.

## 5. Compare fold by fold
Every model was scored on the **same five folds**, so we can put their scores side by side, fold by fold.

In [ ]:
folds

**What we see:** fold 2 is the hardest fold for **every** model, even the dummy. It contains the most expensive sale in the data (record 271, price 117.5), and its prices vary more than in the other folds. A hard fold is hard for all models, so comparing models **within** each fold is fairer than comparing single numbers.

Is "all six inputs" better than "distance only" in every fold, or only on average? Subtract fold by fold:

In [ ]:
gain = folds.loc["distance only"] - folds.loc["all six inputs"]
print("MAE gain of all six inputs over distance only:", gain.round(2).tolist())
print("folds won:", int((gain > 0).sum()), "of", len(gain))

**What we see:** all six inputs win **in all five folds**, by 0.44 to 1.47. A gain that appears in every fold is much stronger evidence than a gain in the average alone.

A useful rule for the rest of the week: **be careful with a difference that is smaller than the fold-to-fold spread (about 0.9–1.0 here) and does not appear in most folds.**

## 6. What do the coefficients say?
Cross-validation fitted five temporary models only to score them. To read coefficients, we fit the chosen model **once on all training rows**.

Each coefficient is the change in predicted price for **one unit** of that input, **holding the other included inputs fixed**.

In [ ]:
six = models["all six inputs"].fit(X_train, y_train)
units = ["per year", "per year of age", "per metre", "per store", "per degree", "per degree"]
pd.DataFrame({"input": features, "unit": units, "coefficient": six.named_steps["model"].coef_}).round(5)

**What we see:**
- One more store nearby: +1.09, holding the other inputs fixed.
- Distance: −0.00476 per metre, or −0.476 per 100 m. In Part 1, distance alone gave −0.741 per 100 m. The value changed because now other inputs (stores, location, age) share the explanation.
- Latitude: +229 per degree. It looks huge only because one degree of latitude is about 111 km.

So raw coefficients **cannot be ranked**: metres, years, degrees, and store counts are different units.

To compare them, we **standardise** every input with `StandardScaler` (Week 2): subtract the training mean and divide by the training standard deviation. Then each coefficient means the change in price for **one standard deviation** of that input.

Standardising changes the units of the coefficients, not the predictions. We check that the CV MAE is unchanged:

In [ ]:
scaled = Pipeline([("inputs", uses(features)), ("scale", StandardScaler()), ("model", LinearRegression())])
scaled_table, _ = evaluate({"all six inputs, standardised": scaled})
print("CV MAE without scaling:", table.loc["all six inputs", "MAE"])
print("CV MAE with scaling:   ", scaled_table["MAE"].iloc[0])

In [ ]:
scaled.fit(X_train, y_train)
per_sd = pd.Series(scaled.named_steps["model"].coef_, index=features)
per_sd.reindex(per_sd.abs().sort_values(ascending=False).index).round(3).to_frame("change per 1 SD")

**What we see:** the same MAE, 6.545. Distance has the largest standardised coefficient (−5.79 per SD), then store count (+3.22) and house age (−3.06).

This ranks **associations inside this model**. It does not prove that distance **causes** the largest price change. The inputs are also related to each other, so each coefficient depends on which other inputs are in the model.

## 7. Where does the model miss?
A good score can hide **systematic** errors: errors that depend on some input.

To look at errors without using the test rows, we need predictions for training rows made by a model that **did not see those rows**.

**New tool — `cross_val_predict(model, X, y, cv=cv)`.** It uses the same five folds. Each row gets its prediction from the fold model that did not see it. These are **out-of-fold** predictions.

In [ ]:
oof = cross_val_predict(models["all six inputs"], X_train, y_train, cv=cv)
check = train[["record_id", "distance_to_mrt_m"]].copy()
check["residual"] = y_train - oof
check.head().round(2)

Individual residuals are noisy. To see a pattern, we group the sales by distance and average the residuals in each group.

**New tool — `pd.cut(values, edges)`** puts each value into a band between chosen boundaries. We use six distance bands, from "up to 300 m" to "3 to 7 km". The bands contain different numbers of sales; the table shows how many.

In [ ]:
distance_edges = [0, 300, 600, 1000, 2000, 3000, 7000]   # band boundaries in metres
check["distance band"] = pd.cut(check["distance_to_mrt_m"], distance_edges)
bands = check.groupby("distance band", observed=True).agg(
    rows=("residual", "size"), band_centre=("distance_to_mrt_m", "median"), mean_residual=("residual", "mean"))
bands.round(2)

If the model had no systematic error, every band mean would be close to zero. The plot shows the individual residuals and the band means (plotted at the middle distance of each band):

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(check["distance_to_mrt_m"], check["residual"], alpha=.45, label="out-of-fold residual")
ax.plot(bands["band_centre"], bands["mean_residual"], color="#00798C", marker="o", linewidth=2.5, label="band mean")
ax.axhline(0, color="#D1495B")
ax.set(title="Residuals of the six-input model", xlabel="distance to MRT (metres)", ylabel="actual − predicted")
ax.legend(); plt.show()

**What we see:** the band means are **not** scattered around zero. They form a U:
- up to 300 m: +4.72, so the model predicts **too low** close to stations;
- 1 to 3 km: −6.71 and −5.31, so it predicts **too high** at medium distances;
- beyond 3 km: +6.18, **too low** again.

This is the **curve** we saw in Part 1's scatter plot. The model uses distance as a straight line, but price falls steeply near stations and then flattens. A straight line cannot bend, so it is wrong in a systematic way at both ends and in the middle. Part 3 tests a remedy.

One point lies far above the others (+77). This is record 271, the 117.5 sale; we will look at it in Part 3.

## 8. Your turn
Add one candidate of your own to `models`: for example distance and stores only, or all inputs except `longitude`. Then run `evaluate(models)` again and compare your candidate with "all six inputs" fold by fold. Is the difference larger than the fold-to-fold spread? Does it appear in most folds?

In [ ]:
# models["my candidate"] = Pipeline([("inputs", uses([...])), ("model", LinearRegression())])
# table, folds = evaluate(models)
# display(table)
# display(folds)

### What you learned
- A single test score is one draw; the same model varied from 6.29 to 8.96 MAE across folds.
- `cross_validate` compares models on the same folds; flip the sign of error scores.
- RMSE weights large misses more; R² compares with predicting the mean and is not a percentage of correct answers.
- Fold-by-fold comparison shows whether a gain is consistent.
- Raw coefficients carry units; standardised coefficients can be compared, but neither proves causation.
- Out-of-fold residuals reveal a curved pattern with distance.
- **The test rows have not been used.**

**Sources:** AITU Lecture 3, slides 13 and 35–37; ISLP, physical PDF page 214 (k-fold cross-validation); scikit-learn user guide, cross-validation and regression metrics.